<a href="https://colab.research.google.com/github/david-levin11/Verification_Notebooks/blob/main/PressureGradientCompare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Gradient Compare**
<br/>
Description--Script will take a pressure gradient time series from two sites and compare it to the wind speed and gust time series from a third site.

- David Levin, Arctic Testbed & Proving Ground, Anchorage Alaska

##**1 - Install Python Packages**
This will manage the installation of the packages we need. You only need to run this once.

In [ ]:
# @title
import requests
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import urllib.parse


##**2 - Enter site data and run script**

In [ ]:
# ------------- USER CONFIG -------------
API_TOKEN = "" #@param {"type":"string"}

# Two stations for SLP gradient (replace with your station IDs)
SLP_SITE_1 = "PASI" #@param {"type":"string"}
SLP_SITE_2 = "PAKT" #@param {"type":"string"}

# Third station for wind
WIND_SITE = "LCNA2" #@param {"type":"string"}

# Time window (UTC)
# Example: past 24 hours
START_DATE = "2021-01-01" #@param {"type":"date"}
END_DATE = "2026-01-01" #@param {"type":"date"}
END_TIME = datetime.strptime(END_DATE, "%Y-%m-%d")
START_TIME = datetime.strptime(START_DATE, "%Y-%m-%d")

# variables / units
SLP_VAR = "sea_level_pressure"      # Synoptic variable for station pressure/sealevel – see note below
WIND_VARS = ["wind_direction","wind_speed", "wind_gust"]

#@markdown Below is for finding high and low values for wind speeds at or above certain pressure differences
FIND_EXTREMES = True #@param {"type": "boolean"}
GRADIENT_EXTREMES = -12 #@param {"type": "number"}

# ------------- HELPER FUNCTIONS -------------


def format_time(dt: datetime) -> str:
    """Format datetime for Synoptic API (YYYYmmddHHMM)."""
    return dt.strftime("%Y%m%d%H%M")


def synoptic_timeseries_request(station, vars_list, start, end, token):
    """
    Call Synoptic Time Series endpoint for a single station and list of variables.
    Returns a tidy pandas DataFrame indexed by datetime.
    """
    base_url = "https://api.synopticdata.com/v2/stations/timeseries"

    # Use special units for pressure, default to english otherwise
    if "sea_level_pressure" in vars_list:
        units = "english,pres|mb"  # keep everything else english, pressure in mb
    else:
        units = "english"

    params = {
        "stid": station,
        "start": format_time(start),
        "end": format_time(end),
        "token": token,
        "vars": ",".join(vars_list),
        "obtimezone": "utc",
        "units": units,
        "hfmetars": "0",
    }

    url = f"{base_url}?{urllib.parse.urlencode(params)}"
    print(f"Requesting: {url}")

    r = requests.get(url, timeout=30)
    r.raise_for_status()
    data = r.json()

    if data.get("SUMMARY", {}).get("RESPONSE_CODE") != 1:
        raise RuntimeError(f"Synoptic error: {data.get('SUMMARY')}")

    station_data = data["STATION"][0]
    times = pd.to_datetime(station_data["OBSERVATIONS"]["date_time"])

    df = pd.DataFrame(index=times)

    # Extract each requested variable if present
    obs = station_data["OBSERVATIONS"]
    # Uncomment this once to see what’s available:
    # print("OBS keys:", obs.keys())

    for v in vars_list:
        obsv = f"{v}_set_1"
        if obsv in obs:
            df[v] = pd.to_numeric(obs[obsv], errors="coerce")
        else:
            print(f"Warning: variable '{obsv}' not in observations for {station}")

    df.sort_index(inplace=True)
    return df



# ------------- MAIN WORKFLOW -------------
def main():
    # 1. Get SLP (or station pressure) for two sites
    df_slp1 = synoptic_timeseries_request(SLP_SITE_1, [SLP_VAR], START_TIME, END_TIME, API_TOKEN)
    df_slp2 = synoptic_timeseries_request(SLP_SITE_2, [SLP_VAR], START_TIME, END_TIME, API_TOKEN)

    print("SLP data head:")
    print(df_slp1.head())
    print(df_slp2.head())

    # 2. Get wind data for third site
    df_wind = synoptic_timeseries_request(WIND_SITE, WIND_VARS, START_TIME, END_TIME, API_TOKEN)
    print("Wind data head:")
    print(df_wind.head())

    # Quick empty checks
    if df_slp1.empty or df_slp2.empty or df_wind.empty:
        print("One or more dataframes are empty. Check variable names, stations, or time range.")
        return

    # 3. Combine SLP sites and compute gradient
    df_slp1 = df_slp1.rename(columns={SLP_VAR: f"{SLP_VAR}_{SLP_SITE_1}"})
    df_slp2 = df_slp2.rename(columns={SLP_VAR: f"{SLP_VAR}_{SLP_SITE_2}"})

    # SLP stations should share timestamps, so inner join is fine here
    df_slp = df_slp1.join(df_slp2, how="inner")

    df_slp["pressure_gradient"] = (
        df_slp[f"{SLP_VAR}_{SLP_SITE_1}"] - df_slp[f"{SLP_VAR}_{SLP_SITE_2}"]
    )

    # 4. Align SLP gradient with wind using nearest-time merge
    # Make sure everything is sorted
    df_slp = df_slp.sort_index()
    df_wind = df_wind.sort_index()

    # Use wind timestamps as the "primary" time axis
    df_combined = pd.merge_asof(
        df_wind,
        df_slp,
        left_index=True,
        right_index=True,
        direction="nearest",
        tolerance=pd.Timedelta("30min"),  # adjust if you want tighter/looser matching
    )

    # Drop rows where we couldn't find a nearby SLP obs
    df_combined = df_combined.dropna(subset=["pressure_gradient"])

    print("Combined dataframe head:")
    print(df_combined.head())
    print("Combined dataframe tail:")
    print(df_combined.tail())

    if df_combined.empty:
        print("Combined dataframe is empty after merge_asof; try increasing the tolerance.")
        return


    # 6. Scatter plot: gradient vs wind speed
    plt.figure(figsize=(6, 6))
    plt.scatter(df_combined["pressure_gradient"], df_combined["wind_speed"], alpha=0.7)
    plt.xlabel("Pressure Gradient (mb)")
    plt.ylabel("Wind Speed")
    plt.title(f"Pressure Gradient vs Wind Speed at {WIND_SITE}")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    if FIND_EXTREMES:
      # Condition: pressure gradient > 8 mb
      if GRADIENT_EXTREMES >= 0:
        mask = df_combined["pressure_gradient"] > GRADIENT_EXTREMES
        df_high_grad = df_combined[mask]
        df_high_grad.to_csv("high_grad.csv")
        print(f"Number of points with gradient > {GRADIENT_EXTREMES} mb: {len(df_high_grad)}")
      else:
        mask = df_combined["pressure_gradient"] < GRADIENT_EXTREMES
        df_high_grad = df_combined[mask]
        df_high_grad.to_csv("high_grad.csv")
        print(f"Number of points with gradient < {GRADIENT_EXTREMES} mb: {len(df_high_grad)}")

      if df_high_grad.empty:
          print(f"No data points with pressure_gradient > {GRADIENT_EXTREMES} mb.")
      else:
          # Highest wind speed when gradient > 8
          idx_max = df_high_grad["wind_speed"].idxmax()
          max_row = df_high_grad.loc[idx_max]

          # Lowest wind speed when gradient > 8
          idx_min = df_high_grad["wind_speed"].idxmin()
          min_row = df_high_grad.loc[idx_min]

          print(f"\n🌬 Highest wind speed for pressure_gradient > {GRADIENT_EXTREMES} mb:")
          print(f"  Time:        {idx_max}")
          print(f"  Wind speed:  {max_row['wind_speed']}")
          print(f"  Wind gust:   {max_row.get('wind_gust', float('nan'))}")
          print(f"  Gradient:    {max_row['pressure_gradient']} mb")

          print(f"\n🍃 Lowest wind speed for pressure_gradient > {GRADIENT_EXTREMES} mb:")
          print(f"  Time:        {idx_min}")
          print(f"  Wind speed:  {min_row['wind_speed']}")
          print(f"  Wind gust:   {min_row.get('wind_gust', float('nan'))}")
          print(f"  Gradient:    {min_row['pressure_gradient']} mb")




if __name__ == "__main__":
    main()